In [58]:
import pandas as pd
import re
import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
from sklearn.model_selection import train_test_split
from tqdm.notebook import tqdm
import matplotlib.pyplot as plt


In [59]:
df1= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_remaining.csv")
df2=pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/tripadvisor_hotel_reviews.csv")


In [60]:
import pandas as pd

# Load the datasets
df1 = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_remaining.csv")
df2 = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/tripadvisor_hotel_reviews.csv")

# Optional: check columns to ensure they match
print("Columns in df1:", df1.columns)
print("Columns in df2:", df2.columns)

# If column names differ, rename columns in df2 to match df1
# For example, if df2 has columns ['Review_Text', 'Rating_Score']
# df2.rename(columns={'Review_Text': 'Review', 'Rating_Score': 'Rating'}, inplace=True)

# Combine datasets
combined_df = pd.concat([df1, df2], ignore_index=True)

# Save combined dataset
combined_path = r"C:/Users/MY PC/OneDrive/Desktop/retrain/combined_reviews.csv"
combined_df.to_csv(combined_path, index=False)

print(f"✅ Combined dataset saved at {combined_path} with shape {combined_df.shape}")


Columns in df1: Index(['Rating', 'Review'], dtype='object')
Columns in df2: Index(['Review', 'Rating'], dtype='object')
✅ Combined dataset saved at C:/Users/MY PC/OneDrive/Desktop/retrain/combined_reviews.csv with shape (389070, 2)


In [61]:
df= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/combined_reviews.csv")

In [62]:
df = df.drop_duplicates()

In [63]:
df.shape


(389070, 2)

In [65]:
rating_counts = df['Rating'].value_counts().sort_index()
rating_counts

Rating
1     35196
2     18835
3     25688
4     54581
5    254770
Name: count, dtype: int64

In [66]:
print("🔍 Checking for missing values...")
print(df.isnull().sum())

🔍 Checking for missing values...
Rating    0
Review    0
dtype: int64


In [67]:
# Check if any reviews are repeated
duplicate_reviews = df[df.duplicated(subset=['Review'], keep=False)]

print("🔍 Total duplicate reviews:", df.duplicated(subset=['Review']).sum())
duplicate_reviews.head()


🔍 Total duplicate reviews: 0


,Rating,Review


In [68]:
df = df.drop_duplicates(subset=['Review', 'Rating'])
print("✅ Duplicates removed. New shape:", df.shape)


✅ Duplicates removed. New shape: (389070, 2)


In [69]:
df.head()

,Rating,Review
0,1,! think I just ruined my dish. open this and p...
1,1,!!! PRODUCT HAS NOTHING TO DO WITH MARVEL!!! F...
2,1,"""1/3 less salt"" is more of a gimmick than a he..."
3,1,"""2% Kopi Luwak"" That's the key part in the de..."
4,1,"""4"" and ""flavor"" are misleading. They all tast..."


In [70]:
TARGET = 5000


balanced_df = df.groupby('Rating', group_keys=False).apply(
    lambda x: x.sample(TARGET, replace=True if len(x) < TARGET else False)
)


balanced_df = balanced_df.sample(frac=1, random_state=42).reset_index(drop=True)

after_counts = balanced_df['Rating'].value_counts().sort_index()
print(balanced_df['Rating'].value_counts())

Rating
2    5000
5    5000
3    5000
1    5000
4    5000
Name: count, dtype: int64


C:\Users\MY PC\AppData\Local\Temp\ipykernel_16940\1183090266.py:4: DeprecationWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  balanced_df = df.groupby('Rating', group_keys=False).apply(


In [72]:
  balanced_df.to_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/balanced_reviews.csv", index=False)

In [73]:
df= pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/balanced_reviews.csv")

In [74]:
import pandas as pd

# Load the dataset
df = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/reviews_imbalanced.csv")

# Find rows where 'Review' is NaN
nan_reviews = df[df['Review'].isna()]

# Show count and examples
print(f"Total reviews with NaN text: {nan_reviews.shape[0]}")
print(nan_reviews.head())


Total reviews with NaN text: 0
Empty DataFrame
Columns: [Rating, Review]
Index: []


In [75]:
print("🔍 Checking for missing values...")
print(df.isnull().sum())

🔍 Checking for missing values...
Rating    0
Review    0
dtype: int64


In [76]:
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer
import nltk

In [77]:
stop_words = set(stopwords.words('english'))
lemmatizer = WordNetLemmatizer()

def preprocess(text):
    tokens = text.lower().split()                 # lowercase + tokenize
    tokens = [w for w in tokens if w.isalpha()]  # keep only words
    tokens = [w for w in tokens if w not in stop_words]  # remove stopwords
    tokens = [lemmatizer.lemmatize(w) for w in tokens]   # lemmatization
    return " ".join(tokens)


In [78]:
df['Cleaned_Text'] = df['Review'].astype(str).apply(preprocess)

# Check the cleaned text
df[['Review','Cleaned_Text']].head()


,Review,Cleaned_Text
0,"I used this for a month, but not sure that it ...",used sure really help baby started burp stoppe...
1,You can't tell this from the wheat brands. Mix...,tell wheat mix make wonderful moist sometimes ...
2,So I've been looking for this cereal literally...,looking cereal literally every time go store e...
3,This sauce is excellent. Medium hot with smoky...,sauce medium hot smoky undertone distinctive g...
4,"Although this cocoa is called Special Dark, it...",although cocoa called special special dark coc...


In [79]:

df.columns


Index(['Rating', 'Review', 'Cleaned_Text'], dtype='object')

In [80]:

df = df.drop(columns=['Review'])

# Check remaining columns
df.columns


Index(['Rating', 'Cleaned_Text'], dtype='object')

In [81]:

df = df.rename(columns={'Cleaned_Text': 'Review'})


df.columns


Index(['Rating', 'Review'], dtype='object')

In [82]:
# Remove any leading/trailing spaces in column names
df.columns = df.columns.str.strip()

# Check columns again
print(df.columns)


Index(['Rating', 'Review'], dtype='object')


In [83]:
print(df.isna().sum())

Rating    0
Review    0
dtype: int64


In [85]:
#split the data set into 80 for training and 20 for testing

train_df, test_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df['Rating']   
)

print("Train:", train_df.shape)
print("Test :", test_df.shape)

# Save
train_df.to_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Bal_train_dataset.csv", index=False)
test_df.to_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Bal_test_dataset.csv", index=False)


Train: (20000, 2)
Test : (5000, 2)


In [86]:
import pandas as pd

# Load datasets
train_df = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Bal_train_dataset.csv")
test_df  = pd.read_csv(r"C:/Users/MY PC/OneDrive/Desktop/retrain/Bal_test_dataset.csv")

# Check for NaN values in train dataset
print("=== NaN values in Train Dataset ===")
print(train_df.isna().sum())
print(f"Total NaN values in Train Dataset: {train_df.isna().sum().sum()}\n")

# Check for NaN values in test dataset
print("=== NaN values in Test Dataset ===")
print(test_df.isna().sum())
print(f"Total NaN values in Test Dataset: {test_df.isna().sum().sum()}")


=== NaN values in Train Dataset ===
Rating     0
Review    11
dtype: int64
Total NaN values in Train Dataset: 11

=== NaN values in Test Dataset ===
Rating    0
Review    3
dtype: int64
Total NaN values in Test Dataset: 3
